In [2]:
# pip install requests pandas notebook


In [3]:
# Import Libraries
import pandas as pd
import requests
import json

In [4]:
# Understand the API Base URL for Countries API
BASE_URL = "https://restcountries.com/v3.1"

print("Base URL:", BASE_URL)


Base URL: https://restcountries.com/v3.1


In [5]:
#Operation 1: GET All Countries

fields = "name,capital,population,region,subregion,flags,area,languages,currencies,cca2"

# Fetch all countries from the API
response = requests.get(f"{BASE_URL}/all?fields={fields}")

# Check the status code
print("Status Code:", response.status_code)
# 200 means SUCCESS, anything else means something went wrong

Status Code: 200


In [6]:
# Convert the response to JSON (a list of dictionaries)
all_countries = response.json()

print(f"Total countries fetched: {len(all_countries)}")
print("\nFirst country sample keys:", list(all_countries[0].keys()))

Total countries fetched: 250

First country sample keys: ['flags', 'name', 'currencies', 'languages', 'cca2', 'capital', 'region', 'subregion', 'area', 'population']


In [7]:
# Explore a Single Country's Raw Data
# Let's look at the raw data for the first country

print(json.dumps(all_countries[0], indent=2))

{
  "flags": {
    "png": "https://flagcdn.com/w320/zw.png",
    "svg": "https://flagcdn.com/zw.svg",
    "alt": "The flag of Zimbabwe is composed of seven equal horizontal bands of green, yellow, red, black, red, yellow and green, with a white isosceles triangle superimposed on the hoist side of the field. This triangle is edged in black, spans about one-fourth the width of the field and has its base on the hoist end. A yellow Zimbabwe Bird superimposed on a five-pointed red star is centered in the triangle."
  },
  "name": {
    "common": "Zimbabwe",
    "official": "Republic of Zimbabwe",
    "nativeName": {
      "bwg": {
        "official": "Republic of Zimbabwe",
        "common": "Zimbabwe"
      },
      "eng": {
        "official": "Republic of Zimbabwe",
        "common": "Zimbabwe"
      },
      "kck": {
        "official": "Republic of Zimbabwe",
        "common": "Zimbabwe"
      },
      "khi": {
        "official": "Republic of Zimbabwe",
        "common": "Zimbabwe"
  

In [8]:
#Operation 2: Extract Fields and Build a DataFrame
# We'll extract the most useful fields from each country

def extract_country_data(country):
    """Extract selected fields from a country dictionary."""
    
    # Some fields are nested, so we use .get() to safely access them
    # and provide a default value if the field doesn't exist
    
    return {
        "Name":        country.get("name", {}).get("common", "N/A"),
        "Official Name": country.get("name", {}).get("official", "N/A"),
        "Capital":     country.get("capital", ["N/A"])[0] if country.get("capital") else "N/A",
        "Region":      country.get("region", "N/A"),
        "Subregion":   country.get("subregion", "N/A"),
        "Population":  country.get("population", 0),
        "Area (km²)":  country.get("area", 0),
        "Languages":   ", ".join(country.get("languages", {}).values()) if country.get("languages") else "N/A",
        "Currencies":  ", ".join(
                           [v.get("name", "") for v in country.get("currencies", {}).values()]
                       ) if country.get("currencies") else "N/A",
        "Continent":   country.get("continents", ["N/A"])[0],
        "UN Member":   country.get("unMember", False),
        "Independent": country.get("independent", False),
        "Country Code": country.get("cca2", "N/A"),
        "Flag Emoji":  country.get("flag", "N/A"),
    }

In [9]:
# Apply the function to every country and build a DataFrame
records = [extract_country_data(c) for c in all_countries]
df = pd.DataFrame(records)

print(f"DataFrame shape: {df.shape}")  # (rows, columns)
df.head()

DataFrame shape: (250, 14)


,Name,Official Name,Capital,Region,Subregion,Population,Area (km²),Languages,Currencies,Continent,UN Member,Independent,Country Code,Flag Emoji
0,Zimbabwe,Republic of Zimbabwe,Harare,Africa,Southern Africa,17073087,390757.0,"Chibarwe, English, Kalanga, Khoisan, Ndau, Nor...",Zimbabwean dollar,N/A,False,False,ZW,N/A
1,Kiribati,Independent and Sovereign Republic of Kiribati,South Tarawa,Oceania,Micronesia,120740,811.0,"English, Gilbertese","Australian dollar, Kiribati dollar",N/A,False,False,KI,N/A
2,Ghana,Republic of Ghana,Accra,Africa,Western Africa,33742380,238533.0,English,Ghanaian cedi,N/A,False,False,GH,N/A
3,North Korea,Democratic People's Republic of Korea,Pyongyang,Asia,Eastern Asia,25950000,120538.0,Korean,North Korean won,N/A,False,False,KP,N/A
4,Spain,Kingdom of Spain,Madrid,Europe,Southern Europe,49315949,505992.0,"Spanish, Catalan, Basque, Galician",euro,N/A,False,False,ES,N/A


In [10]:
# Operation 3: Search by Country Name
# Search for a specific country by name

country_name = "Germany"
response = requests.get(f"{BASE_URL}/name/{country_name}")

if response.status_code == 200:
    data = response.json()
    result = extract_country_data(data[0])
    print(f"Data for {country_name}:")
    for key, value in result.items():
        print(f"  {key}: {value}")
else:
    print(f"Country not found. Status code: {response.status_code}")

Data for Germany:
  Name: Germany
  Official Name: Federal Republic of Germany
  Capital: Berlin
  Region: Europe
  Subregion: Western Europe
  Population: 83491249
  Area (km²): 357114.0
  Languages: German
  Currencies: euro
  Continent: Europe
  UN Member: True
  Independent: True
  Country Code: DE
  Flag Emoji: 🇩🇪


In [11]:
# Operation 4: Filter by Region
# Fetch all countries in Asia
region = "Asia"
response = requests.get(f"{BASE_URL}/region/{region}")

if response.status_code == 200:
    asia_data = response.json()
    asia_records = [extract_country_data(c) for c in asia_data]
    df_asia = pd.DataFrame(asia_records)
    print(f"Countries in {region}: {len(df_asia)}")
    df_asia.head()
else:
    print("Failed:", response.status_code)

Countries in Asia: 50


In [12]:
# Operation 5: Filter by Language
# Fetch countries that speak Spanish
language = "spanish"
response = requests.get(f"{BASE_URL}/lang/{language}")

if response.status_code == 200:
    lang_data = response.json()
    lang_records = [extract_country_data(c) for c in lang_data]
    df_lang = pd.DataFrame(lang_records)
    print(f"Countries speaking Spanish: {len(df_lang)}")
    df_lang[["Name", "Capital", "Region", "Population"]].head(10)

Countries speaking Spanish: 24


In [13]:
# Operation 6: Search by Capital City
# Search by capital city
capital = "Tokyo"
response = requests.get(f"{BASE_URL}/capital/{capital}")

if response.status_code == 200:
    cap_data = response.json()
    result = extract_country_data(cap_data[0])
    print(f"Country with capital '{capital}':")
    for key, value in result.items():
        print(f"  {key}: {value}")

Country with capital 'Tokyo':
  Name: Japan
  Official Name: Japan
  Capital: Tokyo
  Region: Asia
  Subregion: Eastern Asia
  Population: 123210000
  Area (km²): 377930.0
  Languages: Japanese
  Currencies: Japanese yen
  Continent: Asia
  UN Member: True
  Independent: True
  Country Code: JP
  Flag Emoji: 🇯🇵


In [14]:
# Explore and Analyze the Main DataFrame
# Basic info about our full dataset
print("=== DataFrame Info ===")
print(df.info())

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Basic Statistics ===")
df[["Population", "Area (km²)"]].describe()

=== DataFrame Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Name           250 non-null    object 
 1   Official Name  250 non-null    object 
 2   Capital        250 non-null    object 
 3   Region         250 non-null    object 
 4   Subregion      250 non-null    object 
 5   Population     250 non-null    int64  
 6   Area (km²)     250 non-null    float64
 7   Languages      250 non-null    object 
 8   Currencies     250 non-null    object 
 9   Continent      250 non-null    object 
 10  UN Member      250 non-null    bool   
 11  Independent    250 non-null    bool   
 12  Country Code   250 non-null    object 
 13  Flag Emoji     250 non-null    object 
dtypes: bool(2), float64(1), int64(1), object(10)
memory usage: 24.1+ KB
None

=== Missing Values ===
Name             0
Official Name    0
Capital          0
Region   

,Population,Area (km²)
count,2.500000e+02,2.500000e+02
mean,3.207798e+07,6.010389e+05
std,1.319655e+08,1.912575e+06
min,0.000000e+00,4.900000e-01
25%,2.233542e+05,1.194250e+03
50%,5.279123e+06,6.492950e+04
75%,2.037166e+07,3.841505e+05
max,1.417492e+09,1.709825e+07


In [15]:
# Some Interesting Analysis

# Top 10 most populous countries
print("=== Top 10 Most Populous Countries ===")
print(df.nlargest(10, "Population")[["Name", "Population", "Region"]].to_string(index=False))

print("\n=== Number of Countries per Region ===")
print(df["Region"].value_counts().to_string())

print("\n=== Average Population by Region ===")
print(df.groupby("Region")["Population"].mean().sort_values(ascending=False).apply(lambda x: f"{x:,.0f}").to_string())

=== Top 10 Most Populous Countries ===
         Name  Population   Region
        India  1417492000     Asia
        China  1408280000     Asia
United States   340110988 Americas
    Indonesia   284438782     Asia
     Pakistan   241499431     Asia
      Nigeria   223800000   Africa
       Brazil   213421037 Americas
   Bangladesh   169828911     Asia
       Russia   146028325   Europe
       Mexico   130575786 Americas

=== Number of Countries per Region ===
Region
Africa       59
Americas     56
Europe       53
Asia         50
Oceania      27
Antarctic     5

=== Average Population by Region ===
Region
Asia         94,494,639
Africa       24,787,532
Americas     18,617,496
Europe       13,993,546
Oceania       1,779,988
Antarctic           340


In [16]:
# Save DataFrame to CSV

# Save the full DataFrame to a CSV file
# df.to_csv("countries_data.csv", index=False)
# print("Data saved to countries_data.csv ✅")

# Also save the Asia subset
# df_asia.to_csv("asia_countries.csv", index=False)
# print("Asia data saved to asia_countries.csv ✅")